In [22]:
import csv
import os
from dotenv import load_dotenv
from pymongo import MongoClient

In [ ]:
load_dotenv(dotenv_path=".env.local")

# Get URI
uri = os.getenv("MONGODB_URI")
if not uri:
    raise ValueError("MONGO_URI not found in .env.local")

client = MongoClient(uri)  
db = client['planets']                 
collection = db['planets']            

with open('../data/summary.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    try:
        first_row = next(reader)  # Get only the first row
        doc = dict(first_row)

        # Insert first row into MongoDB
        result = collection.insert_one(doc)
        print(f"Inserted document with _id: {result.inserted_id}")

    except StopIteration:
        print("CSV file is empty, nothing to insert.")


Inserted document with _id: 6920c060eb47b3fb1163770d


In [23]:
load_dotenv(dotenv_path=".env.local")

# Get URI
uri = os.getenv("MONGODB_URI")
if not uri:
    raise ValueError("MONGO_URI not found in .env.local")

client = MongoClient(uri)  
db = client['planets']                 
collection = db['planets']            

with open('../data/summary.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)

    documents = []
    for row in reader: 
        doc = row
        obj = dict(row) 

        # Insert row into MongoDB
        result = collection.insert_one(obj)
        print(f"Inserted document with _id: {result.inserted_id}")



Inserted document with _id: 692231e89e037df691c37529
Inserted document with _id: 692231e99e037df691c3752a
Inserted document with _id: 692231e99e037df691c3752b
Inserted document with _id: 692231e99e037df691c3752c
Inserted document with _id: 692231e99e037df691c3752d
Inserted document with _id: 692231e99e037df691c3752e
Inserted document with _id: 692231e99e037df691c3752f
Inserted document with _id: 692231e99e037df691c37530


In [ ]:
result = collection.update_many(
    {"planet": {"$exists": True}}, 
    {"$rename": { "planet": "name"}}
)

print(f"Modified {result.modified_count} documents")

Modified 0 documents


In [17]:
result = collection.update_many(
    { "gravity_m-s2": {"$exists": True}}, 
    { "$rename": { "gravity_m-s2": "gravity_m_s2"}}
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [18]:
result = collection.update_many(
    { "density_kg-km3": {"$exists": True}}, 
    { "$rename": { "density_kg-km3": "density_kg_km3"}}
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [21]:
result = collection.update_many(
    { "planet": {"$exists": True}}, 
    { "$rename": { "planet": "name"}}
)

print(f"Modified {result.modified_count} documents")

Modified 0 documents


In [26]:
fields = {
    "high_temp_c":"$toDouble", 
    "low_temp_c":"$toDouble", 
    "atmos_pressure_mbar":"$toDouble",
    "atmos_N":"$toDouble",
    "atmos_O":"$toDouble",
    "atmos_CO2":"$toDouble",
    "atmos_CH4":"$toDouble",
    "atmos_H":"$toDouble",
    "gravity_m_s2":"$toDouble"
}

set_stage = {
    field: {op: f"${field}"}
    for field, op in fields.items()
}

result = collection.update_many(
    {}, 
    [
        {"$set": set_stage}
    ]
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [ ]:
# Load environment variables from .env.local
load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")  # Your MongoDB URI
DB_NAME = "planets"
COLLECTION_NAME = "planets"
CSV_FILE = "../data/earth_adjusted_summary.csv"  # Path to your Earth-adjusted CSV

# Connect to MongoDB
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Open the CSV and iterate over each row
with open(CSV_FILE, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        planet_name = row.get("planet").strip()  # remove any leading/trailing spaces
        if not planet_name:
            continue

        # Prepare Earth-adjusted fields
        earth_adjusted_fields = {
            "mass_kg_earthAdjusted": float(row["mass_kg"]),
            "volume_km3_earthAdjusted": float(row["volume_km3"]),
            "density_kg_km3_earthAdjusted": float(row["density_kg_km3"]),
            "gravity_m_s2_earthAdjusted": float(row["gravity_m_s2"]),
            "high_temp_c_earthAdjusted": float(row["high_temp_c"]),
            "low_temp_c_earthAdjusted": float(row["low_temp_c"]),
            "atmos_pressure_mbar_earthAdjusted": float(row["atmos_pressure_mbar"]),
            "atmos_N_earthAdjusted": float(row["atmos_N"]),
            "atmos_O_earthAdjusted": float(row["atmos_O"]),
            "atmos_CO2_earthAdjusted": float(row["atmos_CO2"]),
            "atmos_CH4_earthAdjusted": float(row["atmos_CH4"]),
            "atmos_H_earthAdjusted": float(row["atmos_H"]),
        }

        # Update the document in MongoDB
        result = collection.update_one(
            {"name": planet_name},  # match by the 'name' field
            {"$set": earth_adjusted_fields}
        )

        if result.matched_count == 0:
            print(f"Warning: no document found for planet '{planet_name}'")
        else:
            print(f"Updated planet '{planet_name}' with Earth-adjusted values")


Updated planet 'Mercury' with Earth-adjusted values
Updated planet 'Venus' with Earth-adjusted values
Updated planet 'Earth' with Earth-adjusted values
Updated planet 'Mars' with Earth-adjusted values
Updated planet 'Jupiter' with Earth-adjusted values
Updated planet 'Neptune' with Earth-adjusted values


In [31]:
# Load environment variables from .env.local
load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")  # Your MongoDB URI
DB_NAME = "planets"
COLLECTION_NAME = "planets"
CSV_FILE = "../data/earth_adjusted_summary.csv"  # Path to your Earth-adjusted CSV

# Connect to MongoDB
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Open the CSV and iterate over each row
with open(CSV_FILE, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        planet_name = row.get("planet")  # remove any leading/trailing spaces
        if not planet_name:
            continue

        if planet_name == "Saturn": 
            planet_name = "Saturn "

        if planet_name == "Uranus": 
            planet_name = "Uranus "

        # Prepare Earth-adjusted fields
        earth_adjusted_fields = {
            "mass_kg_earthAdjusted": float(row["mass_kg"]),
            "volume_km3_earthAdjusted": float(row["volume_km3"]),
            "density_kg_km3_earthAdjusted": float(row["density_kg_km3"]),
            "gravity_m_s2_earthAdjusted": float(row["gravity_m_s2"]),
            "high_temp_c_earthAdjusted": float(row["high_temp_c"]),
            "low_temp_c_earthAdjusted": float(row["low_temp_c"]),
            "atmos_pressure_mbar_earthAdjusted": float(row["atmos_pressure_mbar"]),
            "atmos_N_earthAdjusted": float(row["atmos_N"]),
            "atmos_O_earthAdjusted": float(row["atmos_O"]),
            "atmos_CO2_earthAdjusted": float(row["atmos_CO2"]),
            "atmos_CH4_earthAdjusted": float(row["atmos_CH4"]),
            "atmos_H_earthAdjusted": float(row["atmos_H"]),
        }

        # Update the document in MongoDB
        result = collection.update_one(
            {"name": planet_name},  # match by the 'name' field
            {"$set": earth_adjusted_fields}
        )

        if result.matched_count == 0:
            print(f"Warning: no document found for planet '{planet_name}'")
        else:
            print(f"Updated planet '{planet_name}' with Earth-adjusted values")


Updated planet 'Mercury' with Earth-adjusted values
Updated planet 'Venus' with Earth-adjusted values
Updated planet 'Earth' with Earth-adjusted values
Updated planet 'Mars' with Earth-adjusted values
Updated planet 'Jupiter' with Earth-adjusted values
Updated planet 'Saturn ' with Earth-adjusted values
Updated planet 'Uranus ' with Earth-adjusted values
Updated planet 'Neptune' with Earth-adjusted values
